In [1]:
using PyPlot
using JLD2
using Statistics
import PhysicalConstants.CODATA2018: c_0
using Unitful

In [2]:
# pathdir_drop_0 = "Y:/TwoDGas/2026/01/19/0046" # Droplets 0°
# pathdir_stripe_60 = "Y:/TwoDGas/2026/02/06/0000" # Stripes 60°
pathdir_stripe_90 = "Y:/StructuralPhaseTransition/2026/02/11/0001" # Stripes 90°

N = 30
n0 = 1279 #1e1
r = 100
λ = 421e-9
γ = 32.7e6 # In Hz
Γ = 2.02e8 # In Hz
N_atoms_per_stripe = 4e4

ω0 = 2π*ustrip(c_0)/λ

x_crop = [1100:1700;]
y_crop = [1750:2450;]

Isat = Γ*ω0^2 / (12*π*ustrip(c_0)^2); # In photon/m^2/atom

### Load experimental data

In [ ]:
# # Droplets 0°
# @load "Saved_exp_data/Intensity_integrated_droplets_"*join(split(pathdir_drop_0, "/")[3:end], "_")*".jld2" Iatoms_drop Ibkg_drop Idark_drop Mean_atoms Mean_bkg Mean_dark Mean_bkg_atoms Std_atoms Std_bkg Std_dark Std_bkg_atoms
# Iatoms_drop_0, Ibkg_drop_0, Idark_drop_0, Mean_atoms_drop_0, Mean_bkg_drop_0, Mean_dark_drop_0, Mean_bkg_atoms_drop_0, Std_atoms_drop_0, Std_bkg_drop_0, Std_dark_drop_0, Std_bkg_atoms_drop_0 = Iatoms_drop, Ibkg_drop, Idark_drop, Mean_atoms, Mean_bkg, Mean_dark, Mean_bkg_atoms, Std_atoms, Std_bkg, Std_dark, Std_bkg_atoms;

In [ ]:
# # Stripes 60°
# @load "Saved_exp_data/Intensity_integrated_stripes_"*join(split(pathdir_stripe_60, "/")[3:end], "_")*".jld2" Iatoms_stripes Ibkg_stripes Idark_stripes Mean_atoms Mean_bkg Mean_dark Mean_bkg_atoms Std_atoms Std_bkg Std_dark Std_bkg_atoms
# Iatoms_stripe_60, Ibkg_stripe_60, Idark_stripe_60, Mean_atoms_stripe_60, Mean_bkg_stripe_60, Mean_dark_stripe_60, Mean_bkg_atoms_stripe_60, Std_atoms_stripe_60, Std_bkg_stripe_60, Std_dark_stripe_60, Std_bkg_atoms_stripe_60 = Iatoms_stripes, Ibkg_stripes, Idark_stripes, Mean_atoms, Mean_bkg, Mean_dark, Mean_bkg_atoms, Std_atoms, Std_bkg, Std_dark, Std_bkg_atoms;

In [3]:
# Stripe 90
@load "Saved_exp_data/Intensity_integrated_stripes_"*join(split(pathdir_stripe_90, "/")[3:end], "_")*".jld2" Iatoms_stripes Ibkg_stripes Idark_stripes Mean_atoms Mean_bkg Mean_dark Mean_bkg_atoms Std_atoms Std_bkg Std_dark Std_bkg_atoms
Iatoms_stripe_90, Ibkg_stripe_90, Idark_stripe_90, Mean_atoms_stripe_90, Mean_bkg_stripe_90, Mean_dark_stripe_90, Mean_bkg_atoms_stripe_90, Std_atoms_stripe_90, Std_bkg_stripe_90, Std_dark_stripe_90, Std_bkg_atoms_stripe_90 = Iatoms_stripes, Ibkg_stripes, Idark_stripes, Mean_atoms, Mean_bkg, Mean_dark, Mean_bkg_atoms, Std_atoms, Std_bkg, Std_dark, Std_bkg_atoms;

### Load simulations

In [8]:
@load "Solutions_sim/Itot_N_10_Sat_[0.1, 5.1, 10.1, 15.1, 20.1, 25.1, 30.1, 35.1]_decay_rate_up_down.jdl2" sat Itot I_SE_SR_SS I_SR_SR_SS nbr_error_N;

### Plot as a function of Sat measured from the images bkg

In [ ]:
close("all")
fig = subplots() # figsize=(20, 10)
title("Bkg-atoms")

# # Droplets 0°
# errorbar(2*Mean_bkg_drop_0/Isat, Mean_bkg_atoms_drop_0, yerr=Std_bkg_atoms_drop_0, color="green", label="Droplet 0°")

# for i = 1:size(Iatoms_drop_0)[1]
#     for j = 1:size(Iatoms_drop_0)[2]
#         scatter(2*Ibkg_drop_0[i, j]/Isat, Ibkg_drop_0[i, j]-Iatoms_drop_0[i, j], color="g", alpha = 0.1)
#     end
# end


# # Stripes 60°
# line_BEC, = errorbar(2*Mean_bkg_stripe_60/Isat, Mean_bkg_atoms_stripe_60, yerr=Std_bkg_atoms_stripe_60, label="Stripes 60°")

# for i = 1:size(Iatoms_stripe_60)[1]
#     for j = 1:size(Iatoms_stripe_60)[2]
#         scatter(2*Ibkg_stripe_60[i, j]/Isat, Ibkg_stripe_60[i, j] - Iatoms_stripe_60[i, j], color=line_BEC.get_color(), alpha = 0.1)
#     end
# end

# Stripes 90°
line_stripe_90, = errorbar(2*Mean_bkg_stripe_90/Isat, Mean_bkg_atoms_stripe_90, yerr=Std_bkg_atoms_stripe_90, label="Stripes 90°")

for i = 1:size(Iatoms_stripe_90)[1]
    for j = 1:size(Iatoms_stripe_90)[2]
        scatter(2*Ibkg_stripe_90[i, j]/Isat, Ibkg_stripe_90[i, j]-Iatoms_stripe_90[i, j], color=line_stripe_90.get_color(), alpha = 0.1)
    end
end

# Simulations
# factor_prop = N_atoms_per_stripe/N
# factor_prop = Mean_bkg_atoms_drop_0[end-1]/mean([Itot[7, j] for j in 1:r if j ∉ nbr_error_t_N[7]])
factor_prop = Mean_bkg_atoms_stripe_90[17]/mean([Itot[7, j] for j in 1:r if j ∉ nbr_error_t_N[7]])
errorbar(Sat, factor_prop*[mean([Itot[i, j] for j in 1:r]) for i in 1:length(Sat)], yerr=factor_prop*[std([Itot[i, j] for j in 1:r if j ∉ nbr_error_t_N[i]])/sqrt(r-length(nbr_error_t_N[i])) for i in 1:length(Sat)], label="Simulations", color="b")

for i in 1:length(Sat)
    for j in 1:r
        if j ∉ nbr_error_t_N[i]
            scatter(Sat[i], factor_prop*Itot[i, j], alpha=0.1, color="blue")
        end
    end
end

xlabel(L"s")
ylabel(L"I_{img}")

legend()

# pygui(true); show();
pygui(false);

LoadError: UndefVarError: `nbr_error_t_N` not defined in `Main`
Suggestion: check for spelling errors or missing imports.